In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog.silver;

In [0]:
%sql

SELECT * FROM catalog.bronze.hospitals_raw;

In [0]:
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

In [0]:
bronze_table = 'catalog.bronze.hospitals_raw'
silver_table = 'catalog.silver.dim_hospitals'
checkpoint_path = "abfss://data@datastoragea.dfs.core.windows.net/silver/dim_hospitals/checkpoint/"

df_hospitals_bronze = spark.readStream.table(bronze_table)

df_hospitals_clean = (
    df_hospitals_bronze
        .drop(
            "status",
            "_rescued_data",
            "_source_file",
            "_ingest_timestamp",
            "ingestion_date"
        )
        .withColumn("load_timestamp", current_timestamp())
)

def merge_dim_hospitals(batch_df, batch_id):
    batch_df = batch_df.drop("status").dropDuplicates(["hospital_id"])

    if not spark.catalog.tableExists(silver_table):
        # First run — create the table
        (batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table))  
        return
    
    # Load Delta table by name and upsert into Silver
    dim_hospitals = DeltaTable.forName(spark, silver_table)

    (dim_hospitals.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.hospital_id = s.hospital_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    

In [0]:
(   
    df_hospitals_clean.writeStream
        .foreachBatch(merge_dim_hospitals)  # for every batch, merge_dim_hospitals is run
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)

In [0]:
%sql
SELECT * FROM catalog.silver.dim_hospitals;

In [0]:
# Data quality check
from pyspark.sql.functions import col, count, min, max

print("DIM HOSPITALS")
df= spark.read.table("catalog.silver.dim_hospitals")
total = df.count()
print(f"Total rows: {total} (expected: 10)")

dup_id = total - df.select("hospital_id").distinct().count()
print(f"Duplicate hospital_ids: {'OK' if dup_id == 0 else dup_id}")

for c in ["hospital_id", "hospital_name", "hospital_class", "bed_count",
          "network_region_cluster"]:
    n = df.filter(col(c).isNull()).count()
    print(f"  {'OK' if n == 0 else 'CHECK'} {c}: {n} nulls")

print("Hospital class distribution:")
df.groupBy("hospital_class").count().show()

print("Cluster distribution:")
df.groupBy("network_region_cluster").count().show()

df.select(
    min("bed_count").alias("min_beds"),
    max("bed_count").alias("max_beds"),
    min("year_opened").alias("earliest"),
    max("year_opened").alias("latest")
).show()
